In [ ]:
%%capture
!pip install keras-cv

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow" # other options: tensorflow or torch
import json
import keras_cv
import keras
from keras import ops
import tensorflow as tf

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
import joblib

import matplotlib.pyplot as plt

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 47  # Random seed
    neg_sample = 0.01 # Downsample negative calss
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [256, 256]  # Input image size
    epochs = 4 # Training epochs
    batch_size = 128  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [ ]:
ROOT = str(os.getcwd())
BASE_PATH = f"{ROOT}/drive/MyDrive/WAIT"
DATA_PATH = f"{BASE_PATH}/data/new"
MODEL_PATH = f"{BASE_PATH}/models/enhanced/"
PROC_PATH = f"{MODEL_PATH}/enhanced/preprocessed_dataset/"

In [ ]:
# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet')

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet')

In [ ]:
print("Class Distribution Before Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

# Sampling
positive_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
negative_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([positive_df, negative_df], axis=0).sample(frac=1.0)

print("\nCalss Distribution After Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

In [ ]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

In [ ]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

# 📊 | Tabular Features

### Enhanced


In [ ]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS = ["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [ ]:
def build_augmenter():
    # Define augmentations
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]

    # Apply augmentations to random samples
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]

    # Build augmentation layer
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    # Apply augmentations
    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        # Read jpeg image
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)

        # Resize
        image = tf.image.resize(image, size=target_size, method="area")

        # Rescale image
        image = tf.cast(image, tf.float32)
        image /= 255.0

        # Reshape
        image = tf.reshape(image, [*target_size, 3])

        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.experimental.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.cache() if cache else ds
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt = tf.data.Options()
        opt.deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)
    ds = ds.map(augment_fn, num_parallel_calls=AUTO) if augment else ds
    ds = ds.prefetch(AUTO)
    return ds

In [ ]:
# Build Training & Validation Dataset
print("# Training:")
training_features = dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(training_ids, training_validation_hdf5, training_features,
                         training_labels, batch_size=CFG.batch_size,
                         shuffle=True, augment=True)

print("# Validation:")
validation_features = dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(validation_ids, training_validation_hdf5, validation_features,
                         validation_labels, batch_size=CFG.batch_size,
                         shuffle=False, augment=False)

# Testing
print("# Testing:")
testing_features = dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(testing_ids, testing_hdf5,
                        testing_features, batch_size=CFG.batch_size,
                         shuffle=False, augment=False, cache=False)

In [ ]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

In [ ]:
training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
feature_space.adapt(training_ds_with_no_labels)
print(feature_space.get_encoded_features().shape)

In [ ]:
# fs_dir = os.path.join(MODEL_PATH, "model-metadata")
# os.makedirs(fs_dir, exist_ok=True)
# filepath = os.path.join(fs_dir, "feature_space_test.keras")

# feature_space.save(filepath)
# print(f"FeatureSpace saved at: '{filepath}'")

In [ ]:
# Apply Feature Processing

training_ds = training_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

validation_ds = validation_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

testing_ds = testing_ds.map(
    lambda x: {"images": x["images"],
               "features": feature_space(x["features"])}, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
# optmizier, loss & metric
adam = keras.optimizers.Adam(learning_rate=1e-4)
auc = keras.metrics.AUC(name='auc')
bce = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

## Modelling

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode="exp", plot=False)

In [ ]:
model_fpath = os.path.join(MODEL_PATH, 'saved_models', 'best_enhanced_model_modattendance_test.keras')

ckpt_cb = keras.callbacks.ModelCheckpoint(
    model_fpath,
    monitor="val_auc",
    save_best_only=True,
    save_weights_only=False,
    mode="max",
)

#### enhanced model (without modality importance)


In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images":image_input, "features":feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate()([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)

# Compile the model
model.compile(
    optimizer=adam,
    loss=bce,
    metrics=[auc],
)

# Model Summary
model.summary()

### Modality Importance

### Fixed modality scaling (α, β in [0,1]) — Simple approach
After obtaining embeddings **x1** (images) and **x2** (tabular), we scale them with **β = 1 − α** and **α**, respectively.

- If **α = 0** → model uses only images  
- If **α = 0.5** → balanced usage  
- If **α = 1** → model uses only tabular


In [ ]:
# metti 0<alpha<1 per i tabulari e 0<beta<1 per immagini

In [ ]:
# define alpha term (0..1) and enforce symmetry
alpha = tf.constant(0.75, dtype=tf.float32)  # set alpha
beta = 1.0 - alpha

# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images": image_input, "features": feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# apply symmetric modality weights
x1_weighted = keras.layers.Lambda(
    lambda x: x * tf.cast(beta, x.dtype),
    name="image_weighting",
)(x1)

x2_weighted = keras.layers.Lambda(
    lambda x: x * tf.cast(alpha, x.dtype),
    name="tabular_weighting",
)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate(name="concat")([x1_weighted, x2_weighted])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)
model.compile(
    optimizer=adam,
    loss=bce,
    metrics=[auc],
)
model.summary()


### Modality attendance sweep (α grid)
Train the fixed-scaling model across a grid of **α** values (β = 1 − α) and compare validation curves.


In [ ]:
# grid of alpha values (constant increments)
alpha_values = np.round(np.linspace(0.0, 1.0, 6), 2)  # 0.0, 0.2, ..., 1.0
sweep_epochs = 20


def build_fixed_scaling_model(alpha):
    alpha = tf.constant(alpha, dtype=tf.float32)
    beta = 1.0 - alpha

    # Define input layers
    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    # Branch for image input
    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    # Branch for tabular/feature input
    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    # apply symmetric modality weights
    x1_weighted = keras.layers.Lambda(
        lambda x: x * tf.cast(beta, x.dtype),
        name="image_weighting",
    )(x1)
    x2_weighted = keras.layers.Lambda(
        lambda x: x * tf.cast(alpha, x.dtype),
        name="tabular_weighting",
    )(x2)

    # Concatenate both branches
    concat = keras.layers.Concatenate(name="concat")([x1_weighted, x2_weighted])

    # Output layer
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

    # Build model
    model = keras.models.Model(inp, out)
    model.compile(
        optimizer=adam,
        loss=bce,
        metrics=[auc],
    )
    return model


histories = {}
for alpha in alpha_values:
    print(f"Training α={alpha:.2f} (β={1.0-alpha:.2f})")
    model = build_fixed_scaling_model(alpha)
    history = model.fit(
        training_ds,
        epochs=sweep_epochs,
        callbacks=[lr_cb],
        validation_data=validation_ds,
        verbose=CFG.verbose,
        class_weight=class_weights,
    )
    histories[float(alpha)] = history
    keras.backend.clear_session()

# Plot validation curves
plt.figure(figsize=(10, 5))
for alpha, history in histories.items():
    plt.plot(history.history["val_auc"], label=f"α={alpha:.2f}")
plt.title("Validation AUC vs Epochs (α sweep)")
plt.xlabel("epoch")
plt.ylabel("val_auc")
plt.legend()
plt.show()

# Plot best validation AUC vs alpha
best_vals = [max(histories[a].history["val_auc"]) for a in alpha_values]
plt.figure(figsize=(8, 4))
plt.plot(alpha_values, best_vals, marker="o")
plt.title("Best Validation AUC vs α")
plt.xlabel("α")
plt.ylabel("best val_auc")
plt.show()


### Loss regularization -- Refined approach

Here we do not modify the architecture, but add a penalty term to the loss that reduces the dependence on the tabular (or image) branch.

We measure the "dependence" through the L2 norm of the weights connecting the modality to the final head."

In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images":image_input, "features":feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate()([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)

In [ ]:
# intensity of the regularization term
lambda_reg = 1e-4

# get the final dense layer and the concatenation layer
dense_final = model.layers[-1]
concat_layer = [l for l in model.layers if isinstance(l, keras.layers.Concatenate)][0]

# Troviamo l'indice di separazione fra x1 e x2 nella concatenazione, serve a sapere quanti pesi appartengono alla parte tabellare
dim_img = concat_layer.input[0].shape[-1]
dim_tab = concat_layer.input[1].shape[-1]

per-sample regularization

In [ ]:
def weighted_loss(y_true, y_pred):
    # BCE base
    base_loss = keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.02)

    # Penalità sui pesi che connettono la parte tabellare all’output
    W = dense_final.kernel  # matrice [dim_img + dim_tab, 1]
    W_tab = W[dim_img:]     # seleziona solo la parte tabellare
    reg_term = lambda_reg * tf.reduce_sum(tf.square(W_tab))

    return base_loss + reg_term

per-batch regularization

In [ ]:
def weighted_loss(y_true, y_pred):
    base_loss = keras.losses.binary_crossentropy(
        y_true, y_pred, label_smoothing=0.02
    )
    base_loss = tf.reduce_mean(base_loss)  # per-batch mean
    W = dense_final.kernel
    W_tab = W[dim_img:]
    reg_term = lambda_reg * tf.reduce_sum(tf.square(W_tab))
    return base_loss + reg_term

In [ ]:
model.compile(
    optimizer=adam,
    loss=bce,
    metrics=[auc],
)

model.summary()

### Controlled fusion (post‑training weighting)  
This variant replaces the standard merge with a weighted sum of the two modality embeddings:  
`fused = alpha_img * z_img + alpha_tab * z_tab`.  
It lets you manually bias the model toward image or tabular features *after training* by changing `alpha_img` and `alpha_tab` (both non‑trainable).  
Note: `z_img` and `z_tab` must have the same dimensionality, since they are summed.

In [ ]:
class FusionWithAlpha(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.alpha_img = self.add_weight(
            name="alpha_img", shape=(), initializer="ones", trainable=False
        )
        self.alpha_tab = self.add_weight(
            name="alpha_tab", shape=(), initializer="ones", trainable=False
        )
    def call(self, inputs):
        z_img, z_tab = inputs
        return self.alpha_img * z_img + self.alpha_tab * z_tab

In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images": image_input, "features": feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Project image branch to match x2 dim (128)
x1 = keras.layers.Dense(128, activation="linear", name="img_proj")(x1)

# Safe controlled fusion layer
fused = FusionWithAlpha(name="fusion_with_alpha")([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(fused)

# Build model
model = keras.models.Model(inp, out)

# Compile the model
model.compile(
    optimizer=adam,
    loss=bce,
    metrics=[auc],
)
model.summary()

In [ ]:
cbs = [lr_cb, ckpt_cb]

history = model.fit(
    training_ds,
    epochs=4,
    callbacks=cbs,
    validation_data=validation_ds,
    verbose=CFG.verbose,
    class_weight=class_weights,
)

 After training, you can manually adjust them to bias the final prediction toward one modality without retraining

In [ ]:
# After training, adjust weights:
fusion_layer = model.get_layer("fusion_with_alpha")
fusion_layer.alpha_img.assign(0.8)
fusion_layer.alpha_tab.assign(1.2)

### Adaptive gating

In [ ]:
# # Per evitare che il gate diventi troppo deterministico (cioè sempre 0 o 1),
# # possiamo aggiungere un termine di entropia alla loss

# def gated_loss(y_true, y_pred):
#     base_loss = keras.losses.binary_crossentropy(y_true, y_pred,
#                                                  label_smoothing=0.02)
#     gate_out = model.get_layer("gate_weights").output
#     entropy = -tf.reduce_mean(tf.reduce_sum(gate_out * tf.math.log(gate_out + 1e-6), axis=1))
#     return base_loss + 1e-4 * entropy

In [ ]:
import keras.ops as K

class GateEntropyLoss(keras.layers.Layer):
    def __init__(self, weight=1e-4, **kwargs):
        super().__init__(**kwargs)
        self.weight = weight

    def call(self, gate_out):
        entropy = -K.mean(K.sum(gate_out * K.log(gate_out + 1e-6), axis=1))
        self.add_loss(-self.weight * entropy)  # maximize entropy
        return gate_out

    def compute_output_shape(self, input_shape):
        return input_shape


class GateEntropyLogger(keras.callbacks.Callback):
    def __init__(self, ds, steps=1):
        super().__init__()
        self.ds = ds
        self.steps = steps

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        gate_model = keras.Model(self.model.inputs, self.model.get_layer("gate_weights").output)

        ents = []
        it = iter(self.ds)
        for _ in range(self.steps):
            x, _ = next(it)
            w = gate_model(x)  # (B, 2)
            ent = -tf.reduce_mean(tf.reduce_sum(w * tf.math.log(w + 1e-6), axis=1))
            ents.append(ent.numpy())

        logs["val_gate_entropy"] = float(np.mean(ents))
        print(f" - val_gate_entropy: {logs['val_gate_entropy']:.4f}")

In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images": image_input, "features": feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)
x1 = keras.layers.Dense(256, activation="selu", name="img_emb")(x1)

# Branch for tabular input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)
x2 = keras.layers.Dense(256, activation="selu", name="tab_emb")(x2)

# ADAPTIVE GATE
# Combine the two embeddings and calculate dynamic weights
fusion_input = keras.layers.Concatenate(name="fusion_input")([x1, x2])

# MLP that produces two normalized weights via softmax
gate = keras.layers.Dense(64, activation="relu")(fusion_input)
gate = keras.layers.Dense(2, activation="softmax", name="gate_weights")(gate)

# attach aux loss + metric
gate = GateEntropyLoss(weight=1e-4, name="gate_entropy_loss")(gate)

# Separate the two weights
w_img = keras.layers.Lambda(lambda w: w[:, 0:1], name="w_img")(gate)
w_tab = keras.layers.Lambda(lambda w: w[:, 1:2], name="w_tab")(gate)

# Apply dynamic weights to the respective modalities
x1_w = keras.layers.Multiply(name="weighted_img")([x1, w_img])
x2_w = keras.layers.Multiply(name="weighted_tab")([x2, w_tab])

# Adaptive fusion (weighted sum)
fused = keras.layers.Add(name="fused")([x1_w, x2_w])

# Final head
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="output")(fused)

model = keras.models.Model(inp, out, name="ISIC_Multimodal_Gated")

In [ ]:
gel_cb = GateEntropyLogger(validation_ds, steps=2)

In [ ]:
# Compile model
model.compile(
    optimizer=adam,
    loss=bce,
    metrics=[auc],
)

model.summary()

In [ ]:
# train model
cbs = [lr_cb, ckpt_cb, gel_cb]

history = model.fit(
    training_ds,
    epochs=20,
    callbacks=cbs,
    validation_data=validation_ds,
    verbose=CFG.verbose,
    class_weight=class_weights,
)

In [ ]:
# get the per batch gating weights
get_gate = keras.models.Model(model.input, model.get_layer("gate_weights").output)

sample_batch = next(iter(validation_ds))
w = get_gate(sample_batch[0])  # shape (batch_size, 2)

print("w_img mean:", np.mean(w[:,0]))
print("w_tab mean:", np.mean(w[:,1]))

#### Sanity cheks

##### Modality Ablation

Evaluates the model's reliance on each modality (image and tabular data).

It quantifies the performance drop (AUC) when a specific modality is either fully removed or partially perturbed, helping to understand its contribution to the overall prediction.

In [ ]:
def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

def ablate_batch(x, mode, strength):
    x = dict(x)
    if mode in ("img", "both"):
        img = tf.cast(x["images"], tf.float32)
        base = tf.ones_like(img) * 0.5  # gray
        x["images"] = (1.0 - strength) * img + strength * base
    if mode in ("tab", "both"):
        feat = tf.cast(x["features"], tf.float32)
        mask = tf.random.uniform(tf.shape(feat)) >= strength
        x["features"] = tf.where(mask, feat, tf.zeros_like(feat))
    return x

def eval_auc_with_ablation(model, ds, mode, strength):
    m = keras.metrics.AUC()
    for x, y in ds:
        x_ab = ablate_batch(x, mode, strength)
        p = model(x_ab, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

base_auc = eval_auc(model, validation_ds)
img_auc  = eval_auc_with_ablation(model, validation_ds, "img", 1.0)
tab_auc  = eval_auc_with_ablation(model, validation_ds, "tab", 1.0)
print("base/img/tab drops:", base_auc-img_auc, base_auc-tab_auc)

##### **Gate Faithfulness**

Assesses how well the dynamically predicted gate weights align with the actual impact of each modality on the model's output.

A higher **Spearman correlation** between a modality's gate weight and its ablation impact suggests that the gate is faithfully representing the modality's importance.

In [ ]:
def spearman(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    ra = a.argsort().argsort().astype(np.float32)
    rb = b.argsort().argsort().astype(np.float32)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb) / (np.linalg.norm(ra)*np.linalg.norm(rb) + 1e-12))

def logit(p, eps=1e-6):
    p = tf.clip_by_value(tf.cast(p, tf.float32), eps, 1-eps)
    return tf.math.log(p/(1-p))

def gate_faithfulness(model, ds, strength=1.0):
    gate_model = keras.Model(model.inputs, model.get_layer("gate_weights").output)

    wimg, wtab, dimg, dtab = [], [], [], []

    for x, y in ds:
        p0 = tf.squeeze(model(x, training=False), -1)
        l0 = logit(p0)

        # img ablation
        xi = ablate_batch(x, "img", strength)
        pi = tf.squeeze(model(xi, training=False), -1)
        di = tf.abs(l0 - logit(pi))

        # tab ablation
        xt = ablate_batch(x, "tab", strength)
        pt = tf.squeeze(model(xt, training=False), -1)
        dt = tf.abs(l0 - logit(pt))

        w = gate_model(x, training=False)  # (B,2)
        wimg.append(w[:,0].numpy()); wtab.append(w[:,1].numpy())
        dimg.append(di.numpy());      dtab.append(dt.numpy())

    wimg = np.concatenate(wimg); wtab = np.concatenate(wtab)
    dimg = np.concatenate(dimg); dtab = np.concatenate(dtab)

    return {
        "spearman(w_img, Δimg)": spearman(wimg, dimg),
        "spearman(w_tab, Δtab)": spearman(wtab, dtab),
    }

print(gate_faithfulness(model, validation_ds, strength=1.0))

##### **Calibration (Brier + ECE)**

Evaluates the model's calibration i.e. how well its predicted probabilities match the true probabilities.

**Brier Score** measures the mean squared difference between predicted probabilities and actual outcomes, indicating overall accuracy and calibration.

**Expected Calibration Error (ECE)** quantifies how far predicted probabilities are from true correctness across different confidence bins, revealing systematic over- or under-confidence.

In [ ]:
def brier_ece(model, ds, n_bins=15):
    y_all, p_all = [], []
    for x, y in ds:
        p = tf.squeeze(model(x, training=False), -1)
        y_all.append(tf.cast(tf.squeeze(y), tf.float32).numpy())
        p_all.append(tf.cast(p, tf.float32).numpy())
    y = np.concatenate(y_all); p = np.concatenate(p_all)

    brier = float(np.mean((p - y)**2))

    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    N = len(p)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if not np.any(m):
            continue
        conf = float(np.mean(p[m]))
        acc  = float(np.mean(y[m]))  # empirical positive rate in bin
        ece += (np.sum(m)/N) * abs(acc - conf)

    return {"brier": brier, "ece": float(ece)}

print(brier_ece(model, validation_ds))